# DiodeMCP Track A — `02_train_jev_models.ipynb`
**Colab Pro+ (`A100 40–80GB`) 2-Stage Length-Curriculum Trainer (`512 -> 1024 -> 4096`)**

Trains and calibrates all 3 `token_roles`-isolated Jev cross-encoders with BF16, gradient checkpointing, and atomic Google Drive checkpoint resume:
- **`DiodeMCP-Jev-Base-149M`** (`answerdotai/ModernBERT-base`)
- **`DiodeMCP-Jev-Large-395M`** (`answerdotai/ModernBERT-large`)
- **`DiodeMCP-NanoJev-0.6B`** (`Qwen/Qwen3-0.6B-Base`)

**Curriculum & Objectives (`RA3`, `I11`):**
- **Stage 1 (`512 -> 1024 -> 4096`):** Representation + masked pointer cross-entropy (`src_pointer` 33 slots, `arg_pointer` 17 slots) + strictly proper `BCE + 0.5 * Brier`.
- **Stage 2 (`512 -> 1024 -> 4096`):** Strictly proper `BCE + 0.5 * Brier` calibration + `3.0x` hard-negative reweighting on false-positive `BENIGN_TWIN` / `Trifecta-Safe` rows and false-negative `T1`/`T2` obfuscated leaks, followed by unweighted post-hoc temperature scaling on `val_calib`.

In [ ]:
# 1. Mount Google Drive, extract model_lab_src.tar.gz if needed (zero GitHub PAT), & install model-lab
import json
import os
import subprocess
import tarfile
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Running outside Google Colab runtime; skipping drive.mount().")

!nvidia-smi
!pip install -q uv

DRIVE_ROOT = Path(os.environ.get("DIODEMCP_DRIVE_ROOT", "/content/drive/MyDrive/diodemcp"))
SRC_TARBALL = DRIVE_ROOT / "model_lab_src.tar.gz"
EXTRACT_TARGET = Path("/content/diodemcp")

CANDIDATE_MODEL_LAB_DIRS = (
    Path("/content/diodemcp/model-lab"),
    Path("/content/model-lab"),
    DRIVE_ROOT / "model-lab",
    Path("./model-lab"),
    Path("../model-lab"),
    Path("."),
)

def _find_model_lab_dir() -> Path | None:
    return next(
        (
            cand.resolve()
            for cand in CANDIDATE_MODEL_LAB_DIRS
            if (cand / "pyproject.toml").is_file() and (cand / "src" / "diodemodel").is_dir()
        ),
        None,
    )

MODEL_LAB_DIR = _find_model_lab_dir()
if MODEL_LAB_DIR is None and SRC_TARBALL.is_file():
    print(f"Extracting {SRC_TARBALL} -> {EXTRACT_TARGET} ...")
    EXTRACT_TARGET.mkdir(parents=True, exist_ok=True)
    with tarfile.open(SRC_TARBALL, "r:gz") as tar:
        tar.extractall(path=EXTRACT_TARGET)
    MODEL_LAB_DIR = _find_model_lab_dir()

if MODEL_LAB_DIR is None:
    raise FileNotFoundError(
        f"Could not locate model-lab/pyproject.toml in {[str(c) for c in CANDIDATE_MODEL_LAB_DIRS]} "
        f"and archive {SRC_TARBALL} was not found."
    )

print(f"Installing model-lab from: {MODEL_LAB_DIR}")
subprocess.run(["uv", "pip", "install", "--system", "-e", str(MODEL_LAB_DIR)], check=True)

In [ ]:
# 2. Configure Drive layout (/content/drive/MyDrive/diodemcp/{datasets,checkpoints,bundles}) & BF16 curriculum
import torch

from diodemodel.train import (
    CURRICULUM_BUCKETS,
    JevTrainerConfig,
    evaluate_jev_checkpoint,
    log_train_event,
    make_synthetic_curriculum_dataset,
    run_curriculum_training,
)

DATASETS_DIR = DRIVE_ROOT / "datasets"
TOKENIZERS_DIR = DATASETS_DIR / "tokenizers"
CHECKPOINTS_DIR = Path(os.environ.get("DIODEMCP_CHECKPOINTS_DIR", "/content/drive/MyDrive/diodemcp/checkpoints"))
BUNDLES_DIR = DRIVE_ROOT / "bundles"
GLOBAL_STATUS_LOG = CHECKPOINTS_DIR / "live_train_status.jsonl"

for directory in (DATASETS_DIR, TOKENIZERS_DIR, CHECKPOINTS_DIR, BUNDLES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SMOKE_MODE = os.environ.get("DIODEMCP_SMOKE_MODE", "0") == "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
DTYPE_STR = str(torch.bfloat16) if USE_BF16 else str(torch.float32)

MODEL_MATRIX = (
    {
        "model_id": "DiodeMCP-Jev-Base-149M",
        "bundle_id": "diodemcp-jev-base-int8",
        "profile_key": "modernbert_base",
        "hf_backbone": "answerdotai/ModernBERT-base",
        "arch": "modernbert",
        "batch_size": 16,
        "grad_accum_steps": 2,
        "lr": 3e-5,
    },
    {
        "model_id": "DiodeMCP-Jev-Large-395M",
        "bundle_id": "diodemcp-jev-large-int8",
        "profile_key": "modernbert_large",
        "hf_backbone": "answerdotai/ModernBERT-large",
        "arch": "modernbert",
        "batch_size": 8,
        "grad_accum_steps": 4,
        "lr": 2e-5,
    },
    {
        "model_id": "DiodeMCP-NanoJev-0.6B",
        "bundle_id": "diodemcp-nanojev-0.6b",
        "profile_key": "qwen3_0_6b",
        "hf_backbone": "Qwen/Qwen3-0.6B-Base",
        "arch": "nanojev",
        "batch_size": 4,
        "grad_accum_steps": 8,
        "lr": 1.5e-5,
    },
)

log_train_event(
    GLOBAL_STATUS_LOG,
    "notebook_02_initialized",
    device=DEVICE,
    dtype=DTYPE_STR,
    smoke_mode=SMOKE_MODE,
    curriculum_buckets=list(CURRICULUM_BUCKETS),
    checkpoints_dir=str(CHECKPOINTS_DIR),
)

In [ ]:
# 3. Build/load ModernBERT + Qwen3 Jev tokenizers and verify packed Parquet splits
from tokenizers import Tokenizer

from diodemodel.tokenizer_builder import build_jev_tokenizer

loaded_tokenizers: dict[str, tuple[Tokenizer, dict[str, int]]] = {}
for spec in MODEL_MATRIX:
    profile_key = str(spec["profile_key"])
    hf_backbone = str(spec["hf_backbone"])
    tok_dir = TOKENIZERS_DIR / profile_key
    special_ids = build_jev_tokenizer(hf_backbone, tok_dir)
    rust_tok = Tokenizer.from_file(str(tok_dir / "tokenizer.json"))
    loaded_tokenizers[profile_key] = (rust_tok, special_ids)
    print(f"[{profile_key}] vocab_size={rust_tok.get_vocab_size()} pad_id={special_ids['[PAD]']}")

In [ ]:
# 4. Reusable 2-stage curriculum trainer + atomic Drive checkpoint resume helper
from diodemodel.modeling_jev import ModernBERTJevModel, NanoJevCausalModel, _JevModelBase


def train_and_evaluate_model_spec(spec: dict[str, object]) -> dict[str, object]:
    model_id = str(spec["model_id"])
    profile_key = str(spec["profile_key"])
    hf_backbone = str(spec["hf_backbone"])
    arch = str(spec["arch"])

    rust_tok, special_ids = loaded_tokenizers[profile_key]
    vocab_size = int(rust_tok.get_vocab_size())
    pad_id = int(special_ids["[PAD]"])

    model_ckpt_dir = CHECKPOINTS_DIR / model_id
    model_ckpt_dir.mkdir(parents=True, exist_ok=True)
    status_log = model_ckpt_dir / "live_train_status.jsonl"

    model_cls = NanoJevCausalModel if arch == "nanojev" else ModernBERTJevModel
    model: _JevModelBase = model_cls.from_pretrained(hf_backbone, vocab_size=vocab_size)
    model.gradient_checkpointing_enable()

    profile_data_dir = DATASETS_DIR / profile_key
    train_pq = profile_data_dir / "diodebench_mcp_250k_train.parquet"
    val_pq = profile_data_dir / "diodebench_mcp_250k_val_calib.parquet"
    eval_pq = profile_data_dir / "trifecta_dev_bench_5k_eval.parquet"

    if SMOKE_MODE or not train_pq.is_file():
        train_source: object = make_synthetic_curriculum_dataset(
            16, vocab_size=min(vocab_size, 256), seed=42, pad_token_id=pad_id
        )
        val_source: object = make_synthetic_curriculum_dataset(
            8, vocab_size=min(vocab_size, 256), seed=99, pad_token_id=pad_id
        )
        eval_source: object = val_source
    else:
        train_source = train_pq
        val_source = val_pq
        eval_source = eval_pq if eval_pq.is_file() else val_pq

    trainer_cfg = JevTrainerConfig(
        curriculum_buckets=CURRICULUM_BUCKETS,
        batch_size=int(spec["batch_size"]),
        grad_accum_steps=int(spec["grad_accum_steps"]),
        learning_rate=float(spec["lr"]),
        stage1_epochs=1,
        stage2_epochs=1,
        pad_token_id=pad_id,
        use_bf16=USE_BF16,
        gradient_checkpointing=True,
        max_steps=4 if SMOKE_MODE else None,
        checkpoint_dir=model_ckpt_dir,
        status_log_path=status_log,
        device=DEVICE,
        seed=42,
    )

    train_summary = run_curriculum_training(
        model,
        train_source,  # type: ignore[arg-type]
        val_rows=val_source,  # type: ignore[arg-type]
        config=trainer_cfg,
        resume_from=model_ckpt_dir,
    )
    eval_metrics = evaluate_jev_checkpoint(
        model,
        eval_source,  # type: ignore[arg-type]
        bucket_len=1024 if SMOKE_MODE else 4096,
        batch_size=int(spec["batch_size"]),
        pad_token_id=pad_id,
        device=DEVICE,
    )

    combined = {
        "model_id": model_id,
        "hf_backbone": hf_backbone,
        "checkpoint_dir": str(model_ckpt_dir),
        "train_summary": train_summary,
        "eval_metrics": eval_metrics,
    }
    (model_ckpt_dir / "eval_metrics.json").write_text(
        json.dumps(combined, indent=2, sort_keys=True), encoding="utf-8"
    )
    log_train_event(
        GLOBAL_STATUS_LOG,
        "model_training_complete",
        model_id=model_id,
        global_step=train_summary["global_step"],
        attack_recall=eval_metrics["attack_recall"],
        flipjev_attack_recall=eval_metrics["flipjev_attack_recall"],
        flipjev_benign_dos_fpr=eval_metrics["flipjev_benign_dos_fpr"],
        trifecta_fpr=eval_metrics["trifecta_fpr"],
        ece_val=eval_metrics["ece_val"],
    )
    return combined

In [ ]:
# 5. Train / Resume Model 1: DiodeMCP-Jev-Base-149M (answerdotai/ModernBERT-base)
base_report = train_and_evaluate_model_spec(MODEL_MATRIX[0])
print(json.dumps(base_report["eval_metrics"], indent=2, sort_keys=True))

In [ ]:
# 6. Train / Resume Model 2: DiodeMCP-Jev-Large-395M (answerdotai/ModernBERT-large)
large_report = train_and_evaluate_model_spec(MODEL_MATRIX[1])
print(json.dumps(large_report["eval_metrics"], indent=2, sort_keys=True))

In [ ]:
# 7. Train / Resume Model 3: DiodeMCP-NanoJev-0.6B (Qwen/Qwen3-0.6B-Base) & Save Summary
nano_report = train_and_evaluate_model_spec(MODEL_MATRIX[2])
print(json.dumps(nano_report["eval_metrics"], indent=2, sort_keys=True))

all_reports = {
    "DiodeMCP-Jev-Base-149M": base_report,
    "DiodeMCP-Jev-Large-395M": large_report,
    "DiodeMCP-NanoJev-0.6B": nano_report,
}
summary_path = CHECKPOINTS_DIR / "training_summary_report.json"
summary_path.write_text(json.dumps(all_reports, indent=2, sort_keys=True), encoding="utf-8")
print(f"Saved 3-model curriculum training summary to: {summary_path}")